# Threat Hunting & Cyber Threat Intelligence (THCTI) Investigation Notebook
## Studi Kasus: WitFoo Precinct 6 Cybersecurity Dataset (Coordinated Campaign 233678)
**Peneliti:** Dzul-Ikram  
**Program Studi:** S1 Teknik Komputer, Fakultas Ilmu Komputer, Universitas Amikom Yogyakarta  
**Dataset:** [witfoo/precinct6-cybersecurity](https://huggingface.co/datasets/witfoo/precinct6-cybersecurity)  
**Live Web Demo:** [https://dzul-ikram.github.io/](https://dzul-ikram.github.io/)  

---
### Deskripsi Investigasi
Notebook ini mendokumentasikan metodologi *Hypothesis-Based Threat Hunting* secara end-to-end:
1. Pemuatan dan ekstraksi telemetri SOC multi-vendor (FireEye HX, Cisco Stealthwatch, Meraki, ASA Firewall).
2. Analisis korelasi multi-sensor pada insiden kunci `25512200-01d5-11ee-8c41-374f8f3a6a1c` (Defiant Pike 233678).
3. Rekonstruksi *Evidence Timeline* kronologis.
4. Pemetaan taktik dan teknik MITRE ATT&CK.
5. Formulasi dan eksekusi *Threat Hunting Queries* untuk mendeteksi *C2 Exfiltration* dan *Lateral Movement*.

In [ ]:
import json
import os
from collections import Counter, defaultdict
from datetime import datetime, timezone

# Lokasi file telemetri yang diekstraksi dari WitFoo Precinct 6
DATA_DIR = r"./analysis_output"
INCIDENTS_FILE = os.path.join(DATA_DIR, "graph_incidents.jsonl")
ATTACK_REPORTS_FILE = os.path.join(DATA_DIR, "graph_attack_reports.jsonl")
SIGNALS_META = os.path.join(DATA_DIR, "signals_metadata.json")

print("Direktori Telemetri:", DATA_DIR)

## 1. Pemuatan Metadata Telemetri Enterprise
Menganalisis volume data, distribusi sensor, dan proporsi label *benign*, *suspicious*, dan *malicious*.

In [ ]:
with open(SIGNALS_META, 'r', encoding='utf-8') as f:
    meta = json.load(f)

print(f"Total Catatan Telemetri: {meta.get('total_records'):,}")
print("Distribusi Label Global:", meta.get('label_distribution'))

signals_info = meta.get('tables', {}).get('signals', {})
print(f"\nLive Signals Rows: {signals_info.get('rows'):,}")
print("Vendor Distribusi Top 5:", dict(list(signals_info.get('vendor_distribution', {}).items())[:5]))
print("Produk Telemetri Top 5:", dict(list(signals_info.get('product_distribution', {}).items())[:5]))

## 2. Investigasi Insiden Target: `25512200-01d5-11ee-8c41-374f8f3a6a1c`
Ekstraksi insiden dengan skor kecurigaan tertinggi (0.859 / 85.9%) yang merepresentasikan upaya *Data Theft* terkoordinasi.

In [ ]:
target_id = "25512200-01d5-11ee-8c41-374f8f3a6a1c"
target_incident = None

with open(INCIDENTS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        inc = json.loads(line)
        if inc.get('id') == target_id:
            target_incident = inc
            break

if target_incident:
    print(f"[FOUND] Incident: {target_incident.get('name')}")
    print(f"  Modus Operandi: {target_incident.get('mo_name')}")
    print(f"  Suspicion Score: {target_incident.get('suspicion_score')}")
    print(f"  Status / Disposition: {target_incident.get('status_name')}")
    print(f"  Organization: {target_incident.get('org')}")
    print(f"  Tools Terlibat:")
    for t in target_incident.get('tools', {}).values():
        print(f"    - {t.get('name')} ({t.get('vendor_name')})")
    print(f"  Entity Nodes:")
    for n in target_incident.get('nodes', {}).values():
        print(f"    - IP: {n.get('ip')} | Internal: {n.get('internal')} | Managed: {n.get('managed')}")
else:
    print("Target incident not found!")

## 3. Ekstraksi Korelasi Multi-Sensor (FireEye HX EDR + Cisco Stealthwatch NDR)
Menampilkan rincian telemetri lead yang memicu alarm keamanan simultan.

In [ ]:
leads = target_incident.get('leads', {})
print(f"Total Detection Leads: {len(leads)}\n")

for lid, ldata in leads.items():
    art = ldata.get('artifact', {})
    prod = ldata.get('product', {}).get('name')
    desc = ldata.get('description')
    src = f"{art.get('clientip')}:{art.get('clientport')}"
    dst = f"{art.get('serverip')}:{art.get('serverport')}"
    act = art.get('action')
    print(f"[LEAD] Sensor: {prod:<38} | Rule: {desc:<35}")
    print(f"       Flow: {src} -> {dst} | Action: {act}")
    print(f"       Raw Syslog: {art.get('message')}\n")

## 4. Analisis Kampanye Multi-Branch (Partisi 233678)
Mengidentifikasi aktivitas paralel di cabang Atlanta, Conyers, Loxley, dan Northbrook pada jendela waktu yang sama.

In [ ]:
campaign_suffix = "233678"
campaign_incidents = []

with open(INCIDENTS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        inc = json.loads(line)
        name = inc.get('name', '')
        if campaign_suffix in name:
            campaign_incidents.append(inc)

print(f"Ditemukan {len(campaign_incidents)} insiden dalam Kampanye Terkoordinasi {campaign_suffix}:\n")
for inc in campaign_incidents:
    first_ts = datetime.fromtimestamp(inc.get('first_observed_at', 0), tz=timezone.utc)
    print(f"- {inc.get('name'):<26} | Score: {inc.get('suspicion_score'):<6} | Leads: {inc.get('lead_count'):<2} | Time: {first_ts}")
    for l in list(inc.get('leads', {}).values())[:1]:
        print(f"  Contoh Aktivitas: {l.get('description')} ({l.get('product',{}).get('name')})")

## 5. Threat Hunting Query 1: Cross-Source Correlation (EDR + Outbound HTTPS Deny)
Query Python / Pandas untuk mendeteksi korelasi antara deteksi domain jahat di EDR dan pemblokiran sesi keluar port 443 di firewall.

In [ ]:
def hunt_cross_source_c2(incident_list, max_window_sec=300):
    findings = []
    for inc in incident_list:
        tools = [t.get('name', '') for t in inc.get('tools', {}).values()]
        rules = [l.get('description', '') for l in inc.get('leads', {}).values()]
        
        has_edr_match = any('domain match' in r.lower() or 'fireeye' in t.lower() for r, t in zip(rules, tools * 5))
        has_ndr_alarm = any('concern' in r.lower() or 'stealthwatch' in t.lower() for r, t in zip(rules, tools * 5))
        
        if has_edr_match and has_ndr_alarm:
            findings.append({
                'incident_id': inc.get('id'),
                'name': inc.get('name'),
                'score': inc.get('suspicion_score'),
                'tools': tools,
                'rules': list(set(rules)),
                'nodes': [n.get('ip') for n in inc.get('nodes', {}).values()]
            })
    return findings

c2_results = hunt_cross_source_c2([target_incident])
print("Hasil Hunting Query 1 (C2 Exfiltration):", json.dumps(c2_results, indent=2))

## 6. Threat Hunting Query 2: Multi-Branch Lateral Probing Detector
Mendeteksi pola pemindaian port non-standar (13731, 8880, 4112, 4010) lintas subnet kantor cabang.

In [ ]:
suspicious_ports = {'13731', '8880', '4112', '4010', 13731, 8880, 4112, 4010}
lateral_sweep_findings = []

for inc in campaign_incidents:
    leads = inc.get('leads', {})
    scanned_nodes = set()
    ports_hit = set()
    for l in leads.values():
        art = l.get('artifact', {})
        dport = art.get('serverport')
        dip = art.get('serverip')
        if dport in suspicious_ports or str(dport) in suspicious_ports:
            ports_hit.add(dport)
            scanned_nodes.add(dip)
            
    if len(scanned_nodes) >= 1 or len(ports_hit) >= 1:
        lateral_sweep_findings.append({
            'incident_name': inc.get('name'),
            'score': inc.get('suspicion_score'),
            'distinct_targets_probed': len(scanned_nodes),
            'ports_detected': list(ports_hit),
            'total_leads': len(leads)
        })

print("Hasil Hunting Query 2 (Lateral Movement Sweeps):")
for f in lateral_sweep_findings:
    print(f) 

## 7. Kesimpulan dan Tindak Lanjut Investigasi
1. **Hipotesis H1 (C2 Beaconing):** Terbukti (*SUPPORTED*). Host `192.168.96.104` melakukan komunikasi C2 ke `100.64.43.120:443`.
2. **Hipotesis H2 (Lateral Probing):** Terbukti (*SUPPORTED*). Ditemukan pemindaian horizontal di 4 kantor cabang regional.
3. **Rekomendasi Utama:** Karantina host terinfeksi, sinkhole domain C2, dan terapkan microsegmentation subnet nirkabel WAP.